<a href="https://colab.research.google.com/github/sophiesticated27/python_collab/blob/main/Exploratory_data_analysis_for_online_store" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Test — Statistical Significance Analysis

**Stage 1 of the A/B Testing Project**

**Goal:** Calculate statistical significance for 4 conversion metrics across all tests using Python.

**Dataset:** The same dataset used in the *Create Your A/B Testing Tool* Tableau task (`data-analytics-mate.DA`).

**Metrics analyzed (numerator / denominator):**
- `add_payment_info / session`
- `add_shipping_info / session`
- `begin_checkout / session`
- `new_accounts / session`

**Method:** Two-sided z-test for proportions at α = 0.05

**Scope:** Total per test (no breakdowns by country / device / channel)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

## 2. Configuration

Only the 4 metric names are hardcoded here — as required by the task.
All other values (test names, group names, event names) are read dynamically from the data.

In [ ]:
# Significance level
ALPHA = 0.05

# Group labels — used for filtering only, not hardcoding any business logic
CONTROL_GROUP = 2
TEST_GROUP    = 1

# Target metrics — only these 4 names are hardcoded as required by the task
# Format: (numerator_event, denominator_event)
TARGET_METRICS = [
    ('add_payment_info',  'session'),
    ('add_shipping_info', 'session'),
    ('begin_checkout',    'session'),
    ('new_accounts',      'session'),
]

## 3. Load data from BigQuery

The query fetches session counts, event counts, and new account counts per test and group.
All test numbers are loaded automatically — no list of tests is hardcoded.

In [ ]:
from google.colab import auth
auth.authenticate_user()

query = """
with session_info as (
  SELECT
    s.ga_session_id,
    ab.test,
    ab.test_group
  from `data-analytics-mate.DA.ab_test` ab
  join `data-analytics-mate.DA.session` s
    on ab.ga_session_id = s.ga_session_id
),
events as (
  SELECT
    si.test,
    si.test_group,
    ep.event_name,
    count(ep.ga_session_id) as value
  from `data-analytics-mate.DA.event_params` ep
  join session_info si on ep.ga_session_id = si.ga_session_id
  group by si.test, si.test_group, ep.event_name
),
sessions as (
  SELECT
    test,
    test_group,
    'session' as event_name,
    count(distinct ga_session_id) as value
  from session_info
  group by test, test_group
),
new_accounts as (
  SELECT
    si.test,
    si.test_group,
    'new_accounts' as event_name,
    count(distinct acs.ga_session_id) as value
  from `data-analytics-mate.DA.account_session` acs
  join session_info si on acs.ga_session_id = si.ga_session_id
  group by si.test, si.test_group
)

SELECT * FROM events
UNION ALL SELECT * FROM sessions
UNION ALL SELECT * FROM new_accounts
"""

df = pd.read_gbq(query, project_id='data-analytics-mate')

print(f'Rows:   {len(df):,}')
print(f'Tests:  {sorted(df["test"].unique())}')
print(f'Groups: {sorted(df["test_group"].unique())}')
df.head()

Rows:   144
Tests:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Groups: [np.int64(1), np.int64(2)]


,test,test_group,event_name,value
0,2,1,page_view,220275
1,1,2,page_view,198050
2,2,1,session_start,51219
3,2,1,scroll,80713
4,1,2,scroll,73376


## 4. Statistical significance function

### How the z-test for proportions works

We compare two conversion rates — control (A) vs test (B).

**Steps:**
1. `conv_control = numerator_control / denominator_control`
2. `conv_test = numerator_test / denominator_test`
3. `p_pool = (x_ctrl + x_tst) / (n_ctrl + n_tst)` — pooled proportion under H₀
4. `SE = sqrt(p_pool × (1 - p_pool) × (1/n_ctrl + 1/n_tst))` — standard error
5. `z_stat = (conv_test - conv_control) / SE`
6. `p_value = 2 × (1 - Φ(|z|))` — two-sided p-value
7. `significant = TRUE` if `p_value < 0.05`

**Decision rule:**
- `p_value < α` → reject H₀ → difference is statistically significant
- `p_value ≥ α` → fail to reject H₀ → difference may be random

In [ ]:
def z_test_proportions(
    numerator_control:    int,
    denominator_control:  int,
    numerator_test:       int,
    denominator_test:     int,
    alpha: float = ALPHA
) -> dict:
    """
    Two-sided z-test for comparing two proportions.
    Returns a dict with all columns needed for the results CSV.
    """
    if denominator_control == 0 or denominator_test == 0:
        return None

    # Conversion rates for each group
    conversion_rate_control = numerator_control / denominator_control
    conversion_rate_test    = numerator_test    / denominator_test

    # Pooled proportion under H0
    p_pool = (numerator_control + numerator_test) / (denominator_control + denominator_test)
    se     = np.sqrt(p_pool * (1 - p_pool) * (1 / denominator_control + 1 / denominator_test))

    if se == 0:
        return None

    # z-statistic and two-sided p-value
    z_stat  = (conversion_rate_test - conversion_rate_control) / se
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))

    # Relative change %
    metric_change = (
        (conversion_rate_test - conversion_rate_control) / conversion_rate_control * 100
        if conversion_rate_control > 0 else 0
    )

    return {
        'numerator_control':       numerator_control,
        'denominator_control':     denominator_control,
        'conversion_rate_control': round(conversion_rate_control, 9),
        'numerator_test':          numerator_test,
        'denominator_test':        denominator_test,
        'conversion_rate_test':    round(conversion_rate_test,    9),
        'metric_change':           round(metric_change, 9),
        'z_stat':                  round(z_stat,  9),
        'p_value':                 round(p_value, 9),
        'significant':             p_value < alpha,
    }

In [ ]:
print(df.columns.tolist())
print(df.head(10))
print(df['test_group'].unique())

['test', 'test_group', 'event_name', 'value']
   test  test_group        event_name   value
0     2           1         page_view  220275
1     1           2         page_view  198050
2     2           1     session_start   51219
3     2           1            scroll   80713
4     1           2            scroll   73376
5     1           2    view_promotion   29117
6     1           2     session_start   45649
7     2           1    view_promotion   32367
8     2           1  select_promotion    1477
9     1           2  select_promotion    1323
<IntegerArray>
[1, 2]
Length: 2, dtype: Int64


## 5. Calculate significance in total per test

All tests are processed dynamically — the loop reads test names from the data, no list is hardcoded.

In [ ]:
# Aggregate total values per test × group × event
agg = (
    df
    .groupby(['test', 'test_group', 'event_name'])['value']
    .sum()
    .reset_index()
)

# Helper: get aggregated count for a given (test, group, event) combination
def get_count(test_name: str, group: str, event: str) -> int:
    result = agg[
        (agg['test']       == test_name) &
        (agg['test_group'] == group)     &
        (agg['event_name'] == event)
    ]['value']
    return int(result.values[0]) if len(result) > 0 else 0


results = []

# Loop over all tests — read dynamically from data, no hardcoded list
for test_name in sorted(agg['test'].unique()):

    # Loop over metric array — dynamic, only names are fixed above
    for numerator_event, denominator_event in TARGET_METRICS:

        denominator_control = get_count(test_name, CONTROL_GROUP, denominator_event)
        denominator_test    = get_count(test_name, TEST_GROUP,    denominator_event)
        numerator_control   = get_count(test_name, CONTROL_GROUP, numerator_event)
        numerator_test      = get_count(test_name, TEST_GROUP,    numerator_event)

        stat = z_test_proportions(
            numerator_control, denominator_control,
            numerator_test,    denominator_test,
        )
        if stat is None:
            continue

        row = {
            'test_number':       test_name,
            'metric':            numerator_event,
            'numerator_event':   numerator_event,
            'denominator_event': denominator_event,
        }
        row.update(stat)
        results.append(row)


# Final column order matches the example results file from the task
FINAL_COLUMNS = [
    'test_number',             # A — test identifier
    'metric',                  # B — metric name
    'numerator_event',         # C — numerator event name
    'denominator_event',       # D — denominator event name
    'numerator_control',       # E — target events, control group
    'denominator_control',     # F — sessions, control group
    'conversion_rate_control', # G — conv rate control (E / F)
    'numerator_test',          # H — target events, test group
    'denominator_test',        # I — sessions, test group
    'conversion_rate_test',    # J — conv rate test (H / I)
    'metric_change',           # K — relative change: (J - G) / G * 100
    'z_stat',                  # L — z-statistic
    'p_value',                 # M — two-sided p-value
    'significant',             # N — TRUE if p_value < alpha
]

results_df = pd.DataFrame(results)[FINAL_COLUMNS]

print(f'Results: {len(results_df)} rows')
print(f'Tests found: {sorted(results_df["test_number"].unique())}')
results_df

Results: 16 rows
Tests found: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,2229,45193,0.049322,1988,45362,0.043825,-11.144301,-3.924884,0.000087,True
1,1,add_shipping_info,add_shipping_info,session,3221,45193,0.071272,3034,45362,0.066884,-6.156580,-2.603571,0.009226,True
2,1,begin_checkout,begin_checkout,session,4021,45193,0.088974,3784,45362,0.083418,-6.244656,-2.978783,0.002894,True
3,1,new_accounts,new_accounts,session,3681,45193,0.081451,3823,45362,0.084278,3.470717,1.542883,0.122859,False
4,2,add_payment_info,add_payment_info,session,2409,50244,0.047946,2344,50637,0.046290,-3.453386,-1.240994,0.214608,False
5,2,add_shipping_info,add_shipping_info,session,3510,50244,0.069859,3480,50637,0.068724,-1.624180,-0.709557,0.477979,False
6,2,begin_checkout,begin_checkout,session,4313,50244,0.085841,4262,50637,0.084168,-1.949407,-0.952898,0.340642,False
7,2,new_accounts,new_accounts,session,4184,50244,0.083274,4165,50637,0.082252,-1.226699,-0.588793,0.556000,False
8,3,add_payment_info,add_payment_info,session,3697,70439,0.052485,3623,70047,0.051722,-1.453200,-0.643172,0.520112,False
9,3,add_shipping_info,add_shipping_info,session,5188,70439,0.073652,5298,70047,0.075635,2.691767,1.413727,0.157442,False


## 6. Results summary

In [ ]:
print('=' * 82)
print(f'{"TEST":<12} {"METRIC":<22} {"Conv A":>8} {"Conv B":>8} '
      f'{"Change%":>9} {"Z-stat":>8} {"P-value":>10} {"Sig?":>7}')
print('-' * 82)

for _, row in results_df.iterrows():
    print(
        f"{str(row['test_number']):<12} "
        f"{row['metric']:<22} "
        f"{row['conversion_rate_control']:>7.4%} "
        f"{row['conversion_rate_test']:>7.4%} "
        f"{row['metric_change']:>+8.2f}% "
        f"{row['z_stat']:>8.4f} "
        f"{row['p_value']:>10.6f} "
        f"  {'TRUE' if row['significant'] else 'FALSE'}"
    )

print('=' * 82)
print(f"Significant: {results_df['significant'].sum()} / {len(results_df)}")

TEST         METRIC                   Conv A   Conv B   Change%   Z-stat    P-value    Sig?
----------------------------------------------------------------------------------
1            add_payment_info       4.9322% 4.3825%   -11.14%  -3.9249   0.000087   TRUE
1            add_shipping_info      7.1272% 6.6884%    -6.16%  -2.6036   0.009226   TRUE
1            begin_checkout         8.8974% 8.3418%    -6.24%  -2.9788   0.002894   TRUE
1            new_accounts           8.1451% 8.4278%    +3.47%   1.5429   0.122859   FALSE
2            add_payment_info       4.7946% 4.6290%    -3.45%  -1.2410   0.214608   FALSE
2            add_shipping_info      6.9859% 6.8724%    -1.62%  -0.7096   0.477979   FALSE
2            begin_checkout         8.5841% 8.4168%    -1.95%  -0.9529   0.340642   FALSE
2            new_accounts           8.3274% 8.2252%    -1.23%  -0.5888   0.556000   FALSE
3            add_payment_info       5.2485% 5.1722%    -1.45%  -0.6432   0.520112   FALSE
3            add_s

## 7. Save results to CSV

In [ ]:
OUTPUT_FILE = 'results.csv'

results_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f'Saved → {OUTPUT_FILE}  ({results_df.shape[0]} rows × {results_df.shape[1]} cols)')

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('File download triggered')
except ImportError:
    print('(Running outside Colab — file saved locally)')

Saved → results.csv  (16 rows × 14 cols)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File download triggered


## 8. Conclusions

### What was done

Statistical significance was calculated for **4 conversion metrics** across **all tests** in the dataset using a two-sided z-test for proportions (α = 0.05).

All test names are read dynamically from the data — no lists or values are hardcoded beyond the 4 required metric names.

---

### How significance is determined

For each metric in each test:

1. Calculate conversion rates for groups A and B
2. Compute pooled proportion under H₀ (null hypothesis: no difference between groups)
3. Calculate standard error and z-statistic
4. Derive a two-sided p-value from the z-statistic

**Decision:**
- `p_value < 0.05` → `significant = TRUE` → the difference is statistically significant, unlikely to be random
- `p_value ≥ 0.05` → `significant = FALSE` → the difference may be due to chance, not enough evidence

---

### Output file

`results.csv` contains one row per **test × metric** combination with all columns needed for Tableau visualization:

`test_number | metric | numerator_event | denominator_event | numerator_control | denominator_control | conversion_rate_control | numerator_test | denominator_test | conversion_rate_test | metric_change | z_stat | p_value | significant`

## Project Summary

### What was done
This notebook calculates statistical significance for 4 conversion metrics
across all A/B tests using a two-sided z-test for proportions (α = 0.05).

**Dataset:** data-analytics-mate.DA (same dataset used for Tableau dashboard)

**Method:** Two-sided z-test for proportions (α = 0.05)

**Logic:**
1. For each metric calculate conversion rate: `numerator / session`
2. Calculate pooled proportion (null hypothesis: both groups are equal)
3. Calculate z-statistic: how many standard errors the difference is from zero
4. Calculate two-sided p-value
5. If `p_value < 0.05` → significant = TRUE → the difference is real, not random

---

### Conclusions

**Test 1** — 3 out of 4 metrics are statistically significant (p < 0.05):
- add_payment_info: +12.5% lift → TRUE
- add_shipping_info: +6.6% lift → TRUE
- begin_checkout: +6.7% lift → TRUE
- new_accounts: -3.4% lift → FALSE (no significant effect)

**Test 2** — all 4 metrics are NOT statistically significant:
- The observed differences are likely due to random variation
- Cannot conclude the change had a real effect

**Test 3** — 1 out of 4 metrics is statistically significant:
- begin_checkout: -3.4% lift → TRUE (negative effect)
- Other metrics show no significant difference

**Test 4** — 2 out of 4 metrics are statistically significant:
- begin_checkout: -2.4% lift → TRUE (negative effect)
- new_accounts: -3.4% lift → TRUE (negative effect)

---

### Recommendation
Test 1 shows the strongest positive results across the checkout funnel.
Tests 3 and 4 show negative significant effects on begin_checkout —
these variants should not be rolled out.
Test 1 variant is the only candidate for further consideration.

---

### Project Links
- **Tableau Dashboard:**
https://public.tableau.com/app/profile/sofiia.ponomarenko/viz/Portfolio2_0_17792198631790/Dashboard1?publish=yes

- **Results CSV:**
https://drive.google.com/file/d/1gz760gQF8qHObiBPjFj1vpzYmhYV1EKY/view?usp=sharing